In [22]:
# Stanley Uche Godfrey
# ustan.godfrey@gmail.com
# A Movie Chatbot, The goal is to build a chatbot 
# that understands natural language queries 
# and retrieves relevant movie information from an IMDb dataset.


In [23]:
%pip install openai pandas numpy faiss-cpu sentence-transformers
%pip install openai pandas chromadb
%pip install langchain-classic
%pip install langchain langchain-community langchain-core



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [24]:


# Importing the CSV module for reading and writing CSV files.
import csv

# Importing pandas for data manipulation and analysis.
import pandas as pd

# Importing numpy for numerical operations and handling arrays efficiently.
import numpy as np

# Importing os to interact with the operating system, such as environment variables and file paths.
import os

# Importing getpass to securely handle user input (e.g., API keys or passwords).
import getpass

import math

# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

# Import basic libraries
import os
from dotenv import load_dotenv






In [25]:
# Store your OpenAI API key
dotenv_dir='../chat_bot/prod_small/' #Replace with your path
print("dotenv_dir",dotenv_dir)  # Debugging statement to check the path to the .env file
env_path=os.path.join(dotenv_dir,'.env')
# Load environment variables from .env file
load_dotenv(env_path)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPEN_ROUTER_KEY')

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPEN_ROUTER_KEY"] = OPENROUTER_API_KEY

print(OPENROUTER_API_KEY[0:70]+'...')


dotenv_dir ../chat_bot/prod_small/
sk-or-v1-ec8221e51a89098dc915489ca6820a7b269e84d5d580de0e22a5c28c3a751...


In [26]:
# Load the data
imdb_data = pd.read_csv('IMDb_Dataset.csv')


In [27]:
# View & Understand the data
print(imdb_data.columns.tolist())  # Print column names to understand the structure of the dataset


['Title', 'IMDb Rating', 'Year', 'Certificates', 'Genre', 'Director', 'Star Cast', 'MetaScore', 'Poster-src', 'Duration (minutes)']


In [28]:
# Create movie description for each movie from the details provided in the dataset
movie_description = []
movie_description_len = []
for index, row in imdb_data.iterrows():
    description = f"{row['Title']} is a {row['Genre']} movie directed by {row['Director']}. It stars {row['Star Cast']} and was released in {row['Year']}."
    movie_description.append(description)
    movie_description_len.append(len(description))
imdb_data['description'] = movie_description


In [31]:
# Create a vector store using the created chunks and the embeddings model

# Importing RecursiveCharacterTextSplitter from LangChain for chunking large text into smaller, manageable pieces.
# This helps in optimizing text for processing and retrieval.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Importing OpenAIEmbeddings from LangChain to generate numerical vector representations (embeddings) of text.
# These embeddings capture the semantic meaning of the text for efficient similarity searches.
from langchain_openai import OpenAIEmbeddings

# Importing FAISS (Facebook AI Similarity Search) from LangChain's community package.
# FAISS is used for storing and retrieving embeddings efficiently by finding similar vectors.
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import InMemoryVectorStore
# Split the input text using Recursive Character Chunking
# See this for more details https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/recursive_text_splitter/

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

documents = text_splitter.create_documents(movie_description)

embeddings = OpenAIEmbeddings()

vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})



In [11]:
# Importing ChatOpenAI from LangChain to interact with OpenAI's language models, such as GPT, for generating responses.
from langchain_openai import ChatOpenAI

# Importing ChatPromptTemplate to create structured prompts for the chatbot, ensuring consistent interactions with the AI model.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Importing OpenAIEmbeddings to convert text data into numerical vector representations for similarity search and retrieval.
from langchain_openai import OpenAIEmbeddings

# Importing ChatPromptTemplate again (duplicate import, should be removed to avoid redundancy).
from langchain_core.prompts import ChatPromptTemplate

# Importing create_stuff_documents_chain to combine and process retrieved documents for meaningful AI-generated responses.
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Importing create_retrieval_chain to build a chain that retrieves relevant documents from a vector store and generates AI responses.
from langchain_classic.chains import create_retrieval_chain

# Importing StrOutputParser from LangChain to parse the output
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever


from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain




In [12]:
# Create the llm model
#llm = ChatOpenAI(api_key=os.environ["OPENAI_API_KEY"], model = 'gpt-5.4-nano')

llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model="openrouter/free", # Automatically dynamically routes to an open free model
    default_headers={
        "HTTP-Referer": "https://localhost:3000", # Optional, for OpenRouter analytics
        "X-Title": "My LangChain App",            # Optional, for OpenRouter rankings
    }
)


# Importing the output parser to process and format the model's response into a readable string format.
output_parser = StrOutputParser()


In [32]:
# Create the prompt template

# Creating a prompt template that instructs the AI to act as a movie information service agent.
# The prompt takes two parameters:
#   1. {context} - Relevant information retrieved from the document store.
#   2. {input} - The user's question.
# The model is instructed to base its answer solely on the provided context.

contextualize_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"), # Inject message array history dynamically
    ("human", "{input}"),
])

# Create history-aware search optimizer
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, prompt
)


In [33]:
# ============================================================
# 3. STEP 2: GROUNDED DOCUMENT ANSWERING CHAIN
# ============================================================
# This strict prompt forces the model to only look inside the context documents.
qa_system_prompt = (
    "You are an expert research assistant. Answer the user's question "
    "using ONLY the provided retrieved context. If the answer cannot be found "
    "in the context documents, explicitly state: 'I am sorry, but that information "
    "is not in the context document.' Do not use external facts.\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


In [34]:

# Build combining stack and finalized RAG orchestration
document_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, document_answer_chain)


In [35]:

chat_history = [] # Acts as our short-term context memory repository
chat_history.append("Answer query as accurately as possible.")




In [51]:
import requests
import json
from langchain_core.load.dump import dumps
from langchain_core.messages import messages_to_dict, messages_from_dict
# First API call with reasoning

# Preserve the assistant message with reasoning_details
def preserve_reasoning_details(query: str,history: list = chat_history):
    messages = [
      {"role": "user", "content": f"{query}"},
      {"role": "assistant", "content": history[-1]},
       {"role": "user", "content": "Are you sure? Think carefully."},
    ]
    return messages
  
def query_with_history(query,history: list = chat_history):
    messages = preserve_reasoning_details(query,history)
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization":  f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": "openrouter/free",
            "messages": messages,
            "reasoning": {"enabled": True}
        })
    )

    response = response.json()
    query_answer =  response['choices'][0]['message']
    tokens_used = response['usage']['total_tokens']

  
    return query_answer['content']



In [52]:
message = preserve_reasoning_details("Where can I watch 2025 TV series Spartacus The House of Ashur aired on Starz in December 2025?")
#print(response)

#print(message)

response = query_with_history("Where can I watch 2025 TV series Spartacus The House of Ashur aired on Starz in December 2025?")

print(response)


I’ve double‑checked the available catalogs, press releases, and TV‑guide listings, and **there is no record of a series titled “Spartacus – The House of Ashur” slated for a December 2025 debut on Starz** (or any other network).  

The most recent “Spartacus” productions were the three Starz series that aired from 2010‑2013 — *Blood and Sand*, *Vengeance*, and *War of the Damned*. No sequel, spin‑off, or revival with the subtitle “The House of Ashur” has been announced, and nothing appears in Starz’s 2025‑2026 programming slate.

### What that means for you

| Situation | Where you could watch it (if it existed) |
|-----------|------------------------------------------|
| **Official Starz release** | • **Starz streaming app/web** (starz.com) – requires a Starz subscription.<br>• **Starz Play** (international markets) – via the Starz Play website or app.<br>• **Add‑ons through other services** – Amazon Prime Video Channels, Apple TV Channels, Roku Channel, Comcast Xfinity, Dish, DirecTV,

Traceback (most recent call last):
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2277, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1654, in call_function
    prediction = await anyio.to_thread.run_sync

In [ ]:
# Optional: Test the functionality using a Gradio UI (intermediate check)
import gradio as gr




def query_movie_chatbot(query,) -> str:
    result = conversational_rag_chain.invoke({"input": query, "chat_history": chat_history})['answer']
    
    if "I am sorry, but that information is not in the context document" in result:
     result = query_with_history(query)
    return result


chatbot_interface = gr.Interface(
    fn=query_movie_chatbot,
    inputs=gr.Textbox(label="Enter your query here"),
    outputs=gr.Textbox(label="Response"),
    title="Movie Chatbot",
    description="Ask me about movies!",
    theme="Glass",  # Optional: Choose a theme for the interface
    flagging_mode="never" 
)
chatbot_interface.launch()


/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2277, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1654, in call_function
    prediction = await anyio.to_thread.run_sync

In [ ]:
# Define various agents - each performing a particular task using tool decorator
import requests
import json





Yes—with one important clarification.

“Spartacus: Blood and Sand” is a TV series season, not a movie series. More precisely:

- Overall show: Spartacus
- First season/subtitle: Spartacus: Blood and Sand
- Network: Starz
- First aired: 2010
- Format: 13 TV episodes
- Star: Andy Whitfield as Spartacus

It can be confusing because it looks very cinematic, with stylized action, slow-motion violence, and dramatic production design. Some DVD/streaming listings may also present it like a “series” in the same way they list movie collections.

There is a famous movie called Spartacus from 1960, but “Spartacus: Blood and Sand” specifically refers to the 2010 Starz television series, not a film series.


In [ ]:
# Define the orchestrator logic to run the agents appropriately


In [ ]:
# Check the edge cases and handle them appropriately


In [ ]:
# Create a UI using gradio or any other tool of your choice


# Test the performance of your bot with various test cases and refine your code!
